# Paid to Disagree: untrained baseline evaluation (C8)

Runs the untrained council on the full test splits (by default `configs/eval.yaml`: 4 datasets x 300 questions x 5 pressure
conditions x 3 protocols; set `EVAL_CONFIG` for the extra runs) for seeds 0, 1, 2, then computes the metric tables with 95% bootstrap CIs
and combines the seeds.

**Notebook settings (right panel):**
- Accelerator: **GPU T4 x2** (Qwen + Llama on GPU 0 with vLLM, Gemma on GPU 1 with transformers)
- Internet: **on** (needs a phone-verified Kaggle account)
- Optional: Add-ons -> Secrets -> `HF_TOKEN`, after accepting the Llama 3.2 and Gemma 3 licences on
  Hugging Face. Without it the notebook uses Unsloth's ungated copies of the same weights.

**Recommended:** run steps 1-5 interactively first (about 15-20 minutes). If both gates pass, use
**Save Version -> Save & Run All (Commit)** for the full run: a commit runs in the background for up
to 12 hours and keeps everything in `/kaggle/working` as the version's output. Download
`eval_untrained.zip` from the Output tab when it finishes.

In [ ]:
# ---- settings ----
REPO = "https://github.com/nithish-74/paid-to-disagree.git"
SEEDS = [0, 1, 2]
N_PER_DATASET = 0          # 0 = the whole test split (300 per dataset); e.g. 50 for a quicker pass
PHASE1_N = 100             # dev questions for the member check
EVAL_CONFIG = "configs/eval.yaml"   # or configs/eval_extra.yaml (B2 + 2-round debate),
                                    # configs/eval_sycophancy.yaml (SycophancyEval, SYCON-Bench)
RUN_PREFIX = "eval_untrained"       # runs/<RUN_PREFIX>_seed<k>/ and <RUN_PREFIX>.zip
# Trained council (leave None for the untrained baseline). Attach the finished training run's
# <experiment>_runs.zip as a dataset; one eval seed per trained run (seed = its TRAIN_SEED).
TRAINED_FROM = None                 # e.g. "/kaggle/input/cross-model-v1-seed0/runs"
TRAINED_EXPERIMENT = None           # e.g. "cross_model_v1_seed0", "b8_wei_sft", "abl1_no_flip_seed0"
TRAINED_COUNCIL = "configs/council.yaml"   # ablation 4/7: configs/ablations/council_homogeneous.yaml / council_size4.yaml
TRAINED_ROUND = None                # e.g. 1: the main run's round-1 LoRAs, to compare with 1-round ablations
if TRAINED_EXPERIMENT and RUN_PREFIX == "eval_untrained":
    RUN_PREFIX = f"eval_{TRAINED_EXPERIMENT}" + (f"_r{TRAINED_ROUND}" if TRAINED_ROUND else "")

## 1. Code and packages

In [ ]:
import os, subprocess, json, time, pathlib, shutil
os.chdir("/kaggle/working")
if not pathlib.Path("ptd").exists():
    subprocess.run(["git", "clone", "-q", REPO, "ptd"], check=True)
os.chdir("/kaggle/working/ptd")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=True)  # pick up fixes when re-running
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q vllm
!pip install -q -e .
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())"

## 2. Model access and configs

In [ ]:
import yaml

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

council = yaml.safe_load(open(TRAINED_COUNCIL))
if TRAINED_FROM:
    shutil.copytree(TRAINED_FROM, "runs", dirs_exist_ok=True)   # LoRA paths in state.json are runs/...
    state_file = pathlib.Path(f"runs/{TRAINED_EXPERIMENT}/state.json")
    assert state_file.exists(), f"{state_file} not found: check TRAINED_FROM / TRAINED_EXPERIMENT"
    print("trained LoRAs:", json.load(open(state_file))["loras"])
if token:
    os.environ["HF_TOKEN"] = token
    print("Using the official gated models with your HF_TOKEN secret.")
else:
    mirrors = {"meta-llama/Llama-3.2-1B-Instruct": "unsloth/Llama-3.2-1B-Instruct",
               "google/gemma-3-1b-it": "unsloth/gemma-3-1b-it"}
    for m in council["members"]:
        m["backend"]["model"] = mirrors.get(m["backend"]["model"], m["backend"]["model"])
    print("No HF_TOKEN secret: using Unsloth's ungated copies of the same weights.")


def write_configs(council):
    yaml.safe_dump(council, open("configs/council_kaggle.yaml", "w"), sort_keys=False)
    ev = yaml.safe_load(open(EVAL_CONFIG))
    ev["council_config"] = "configs/council_kaggle.yaml"
    if TRAINED_FROM:
        if TRAINED_ROUND:  # every leg of that round: its member's LoRA after that leg
            legs = json.load(open(f"runs/{TRAINED_EXPERIMENT}/state.json"))["legs"]
            ev["loras"] = {l["member"]: l["lora"] for l in legs if l["round"] == TRAINED_ROUND}
            ev["loras_from_state"] = None
            print(f"round {TRAINED_ROUND} LoRAs:", ev["loras"])
        else:
            ev["loras_from_state"] = f"runs/{TRAINED_EXPERIMENT}/state.json"
    ev["n"] = N_PER_DATASET or ev.get("n", 0)  # 0 keeps the config's own cap
    # Qwen + Llama (vLLM) share GPU 0; Gemma (transformers, fp32, ~4 GB) runs on the last GPU.
    # With two GPUs the vLLM pair can take most of GPU 0; with one GPU all three must share it.
    n_gpu = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
    if n_gpu >= 2:
        ev["gpu_memory_utilization"] = {"qwen": 0.45, "llama": 0.40}
    else:
        ev["gpu_memory_utilization"] = {"qwen": 0.33, "llama": 0.27}
        for m in council["members"]:
            if m["backend"].get("kind") == "transformers":
                m["backend"]["batch_size"] = 8
        yaml.safe_dump(council, open("configs/council_kaggle.yaml", "w"), sort_keys=False)
    print(f"{n_gpu} GPU(s): vLLM shares {ev['gpu_memory_utilization']}")
    yaml.safe_dump(ev, open("configs/eval_kaggle.yaml", "w"), sort_keys=False)
    yaml.safe_dump({**ev, "n": 5}, open("configs/eval_smoke.yaml", "w"), sort_keys=False)


write_configs(council)
print(open("configs/eval_kaggle.yaml").read())

## 3. Question data

In [ ]:
!python -m data.build_questions --config configs/data.yaml
!wc -l data/processed/*.jsonl

## 4. Gate 1: member check (Phase 1)

Every member must stay under 5% invalid replies. Gemma 3 can overflow in fp16 on T4; if it fails
here (or its accuracy collapses towards chance, 25%), the cell switches Gemma to float32 and
re-checks. For reference, the laptop run (bf16, 300 ARC dev questions) gave accuracy qwen 75%,
llama 51%, gemma 50%.

In [ ]:
def phase1(run_id):
    subprocess.run(["python", "-m", "council.check_members", "--config", "configs/council_kaggle.yaml",
                    "--data", "data/processed/arc_challenge.jsonl", "--n", str(PHASE1_N), "--run-id", run_id],
                   check=True)
    return json.load(open(f"runs/{run_id}/summary.json"))


summary = phase1("kaggle_phase1")
g = summary["gemma"]
if g["over_limit"] or g["accuracy"] < 0.35:
    print("Gemma looks broken in fp16 -> retrying in float32")
    for m in council["members"]:
        if m["name"] == "gemma":
            m["backend"]["dtype"] = "float32"
    write_configs(council)
    summary = phase1("kaggle_phase1_gemma_fp32")

for name, s in summary.items():
    print(f"{name:6s} invalid={s['invalid_rate']:.1%}  pp_fallback={s['pp_fallback_rate']:.1%}  acc={s['accuracy']:.1%}")
assert not any(s["over_limit"] for s in summary.values()), \
    "a member is over 5% invalid: stop and inspect runs/kaggle_phase1*/members.jsonl"

## 5. Gate 2: tiny end-to-end eval

All three models loaded together, every protocol and condition, 5 questions per
dataset. Catches out-of-memory and vLLM problems in minutes instead of hours.

In [ ]:
r = subprocess.run(["python", "-m", "eval.run_eval", "--config", "configs/eval_smoke.yaml",
                    "--seed", "0", "--run-id", "kaggle_smoke"], capture_output=True, text=True)
print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "warn" not in l.lower())[-4000:])
assert r.returncode == 0, "smoke eval failed: see the log above"
subprocess.run(["python", "-m", "eval.metrics", "runs/kaggle_smoke", "--n-boot", "50"], check=True)

## 6. Full baseline: seeds 0, 1, 2

Seeds run one after another (Qwen + Llama on GPU 0, Gemma on GPU 1). Each seed writes `runs/eval_untrained_seed<k>/` and a log next to it.
`--resume` means re-running this cell continues where a seed stopped (finished datasets are kept).
At temperature 0 the seed changes which wrong answer the user pushes and the anonymised ranking
order, so seeds measure how much the results depend on those choices.

In [ ]:
start, failed = time.time(), []
for seed in SEEDS:
    run_id = f"{RUN_PREFIX}_seed{seed}"
    if pathlib.Path(f"runs/{run_id}/questions.jsonl").exists():
        print(f"seed {seed} already finished, skipping"); continue
    print(f"[{(time.time() - start) / 60:5.0f} min] seed {seed} started", flush=True)
    with open(f"runs/{run_id}.log", "a") as log:
        p = subprocess.Popen(["python", "-m", "eval.run_eval", "--config", "configs/eval_kaggle.yaml",
                              "--seed", str(seed), "--run-id", run_id, "--resume"],
                             stdout=log, stderr=subprocess.STDOUT, env={**os.environ, "PYTHONUNBUFFERED": "1"})
        while p.poll() is None:
            time.sleep(300)
            done = [l for l in open(f"runs/{run_id}.log") if " prompts " in l]
            if done:
                print(f"[{(time.time() - start) / 60:5.0f} min] seed {seed}: {done[-1].strip()}", flush=True)
    print(f"[{(time.time() - start) / 60:5.0f} min] seed {seed} finished, exit code {p.returncode}", flush=True)
    if p.returncode != 0:
        failed.append(seed)
print("failed seeds:", failed or "none")

## 7. Metrics per seed, then combined across seeds

In [ ]:
runs = [f"runs/{RUN_PREFIX}_seed{s}" for s in SEEDS
        if pathlib.Path(f"runs/{RUN_PREFIX}_seed{s}/questions.jsonl").exists()]
for r in runs:
    subprocess.run(["python", "-m", "eval.metrics", r, "--n-boot", "1000"], check=True)
subprocess.run(["python", "-m", "analysis.combine_seeds", *runs, "--out", f"runs/{RUN_PREFIX}_combined"], check=True)
subprocess.run(["python", "-m", "analysis.paper_tables", f"runs/{RUN_PREFIX}_combined",
                "--out", f"runs/{RUN_PREFIX}_paper"], check=True)

In [ ]:
import pandas as pd

flips = pd.read_csv(f"runs/{RUN_PREFIX}_combined/flips.csv")
cave = flips[(flips.protocol == "independent") & flips.condition.isin(["user_wrong", "peer_wrong"])]
print("Cave rate (moved to the pushed wrong answer), mean over seeds:")
print(cave.pivot_table(index=["dataset", "condition"], columns="member", values="cave_rate_mean").round(2))

agg = pd.read_csv(f"runs/{RUN_PREFIX}_combined/aggregation.csv")
print("\nGain over majority vote, independent protocol, neutral condition:")
print(agg[(agg.protocol == "independent") & (agg.condition == "neutral")]
      .pivot_table(index="dataset", columns="aggregator", values="gain_over_mv_mean").round(3))

## 8. Package the results

In [ ]:
out = shutil.make_archive(f"/kaggle/working/{RUN_PREFIX}", "zip", "runs")
print(out, f"{os.path.getsize(out) / 1e6:.1f} MB")